<a href="https://colab.research.google.com/github/ssykes-eth/ETH_273-0003-00L/blob/weekend_3/rag_new/rag_tutorial_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a RAG Pipeline at Dunder Mifflin

## Welcome!

You are a software developer at **Dunder Mifflin Paper Company**, Scranton branch. Your boss, Michael Scott, has just walked up to your desk and said:

> *"I need you to build me one of those AI chatbot things. But it has to know OUR stuff, like who the best boss is, what the vacation policy is, and why Dwight is NOT the Assistant Regional Manager. Can you do that by Friday?"*

Challenge accepted. In this notebook, you will build a **RAG pipeline**: a system that retrieves internal company documents and uses them to answer questions with a real LLM.

## What is RAG?

**RAG** stands for **Retrieval-Augmented Generation**. The idea is simple:

- A language model on its own can only answer from what it "memorized" during training.
- **With RAG**, before answering, the system **searches** a collection of documents, finds the relevant ones, and uses them as context.

Think of it like this:
- **Without RAG** = a **closed-book exam**: you answer from memory (you might guess wrong)
- **With RAG** = an **open-book exam**: you search your notes first, then answer (much more reliable!)

### The RAG Pipeline

```
Documents --> Chunk --> Embed --> Store in Vector DB
                                        |
User Question --> Embed --> Retrieve Top-k --> Build Prompt --> LLM --> Answer
```

| Step | What it does | Analogy |
|------|-------------|---------|
| **Chunk** | Split documents into shorter passages | Cutting a book into pages |
| **Embed** | Convert text into a list of numbers (a vector) | Creating a "fingerprint" of the meaning |
| **Store** | Save vectors in a database optimized for similarity search | Filing fingerprints in a cabinet |
| **Retrieve** | Find the most similar passages to the user's question | Pulling the most relevant pages |
| **Augment** | Insert the retrieved passages into the LLM's prompt | Handing the relevant pages to the student |
| **Answer** | The LLM reads the passages and generates an answer | The student reads and answers |

## How to work through this notebook

Most of the code is already written for you. Look for the 🎯 symbol - it marks the **only** lines you need to modify. The exercises are designed as **fill-in-the-blank**: you just need to replace `___` with the correct value. Each 🎯 line tells you exactly what to write.

There are **7 small tasks** in total. Read the surrounding code and comments carefully, the answer is always nearby!

---
## Setup

Run the cells below to install dependencies, import libraries, and configure the LLM.

| Library | What it does in our exercise |
|---------|------------------------------|
| `langchain` | Framework that connects the pieces of our RAG pipeline |
| `chromadb` | Vector database, a "smart filing cabinet" that finds similar documents |
| `fastembed` | Runs a small embedding model locally on your CPU (no API key needed!) |
| `langchain-openai` | Lets us call GPT-4o-mini through the course LLM proxy |

In [1]:
# ── Setup ────────────────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/rag_new')

sys.path.insert(0, '.')

!pip install langchain langchain-text-splitters langchain-openai langchain-community chromadb fastembed --quiet
print("Setup complete!")

Cloning into 'coding-exercises'...
remote: Enumerating objects: 1674, done.
remote: Counting objects: 100% (256/256), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 1674 (delta 147), reused 191 (delta 104), pack-reused 1418 (from 1)
Receiving objects: 100% (1674/1674), 80.81 MiB | 21.44 MiB/s, done.
Resolving deltas: 100% (945/945), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━

In [2]:
import json

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("All libraries loaded!")

All libraries loaded!


In [3]:
# ── LLM Proxy Configuration ──────────────────────────────────────────────
# We use a shared LLM proxy provided for this course.
# Embeddings run locally (no API key needed), only the LLM needs the proxy.

LITELLM_BASE_URL = "https://litellm.sph-prod.ethz.ch"
LITELLM_API_KEY = "sk-KQpp98f7RpzdiAtPNGJZ7w"

# Set env var so ChatOpenAI picks it up automatically
os.environ["OPENAI_API_KEY"] = LITELLM_API_KEY

print("LLM proxy configured!")

LLM proxy configured!


---
## Step 1: Load the Knowledge Base

Every RAG system needs a **knowledge base**, a collection of documents the system can search through. Think of it as the "textbook" that our system will consult during the open-book exam.

Our knowledge base contains internal Dunder Mifflin documents: policies, memos, client info, and employee details. They are stored in a JSON file.

In [4]:
# ── Load documents from JSON ─────────────────────────────────────────────
with open("data/dunder_mifflin_docs.json", "r") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} internal documents.\n")
for doc in documents:
    print(f"  [{doc['id']}] {doc['title']}")

Loaded 13 internal documents.

  [doc_01] Regional Manager
  [doc_02] Assistant to the Regional Manager
  [doc_03] Vacation Policy
  [doc_04] Break Room Rules
  [doc_05] Friday Snack Rotation
  [doc_06] Client Profile - Blue Cross of Pennsylvania
  [doc_07] Reception and Front Desk
  [doc_08] Sales Department Overview
  [doc_09] Accounting Department
  [doc_10] Party Planning Committee
  [doc_11] Office Safety Memo
  [doc_12] Parking Lot Policy
  [doc_13] Dundie Awards


In [5]:
# ── Inspect a few documents ──────────────────────────────────────────────
for doc in documents[:3]:
    print(f"--- {doc['title']} ---")
    print(doc["content"])
    print()

--- Regional Manager ---
Michael Scott is the Regional Manager of the Dunder Mifflin Scranton branch. Michael has been the regional manager for over ten years and is the highest-ranking employee in the Scranton office. As regional manager, Michael reports directly to the corporate office in New York. He is known for his unconventional management style, which includes frequent office parties and motivational speeches.

--- Assistant to the Regional Manager ---
Dwight K. Schrute holds the official title of Assistant to the Regional Manager at Dunder Mifflin Scranton. Despite Dwight's frequent claims, Dwight is NOT the Assistant Regional Manager. Dwight's actual title is Assistant TO the Regional Manager, which is a very different and less important position. Dwight takes his role extremely seriously and Dwight often tries to enforce rules that do not exist.

--- Vacation Policy ---
Dunder Mifflin Scranton employees receive 15 days of paid vacation per year. Unused vacation days cannot be

---
## 🎯 Task 1: Expand the Knowledge Base

Michael insists that the knowledge base includes a very important fact: **he is the best boss in the world.**

Your job: add a new document to the list. It should have:
- A `"title"` (a short name for the document)
- A `"content"` that clearly states Michael Scott is the best boss in the world

Look at the documents above for the format. Your new document should follow the same structure.

In [14]:
# ── Task 1: Add a document about Michael being the best boss ─────────────
#
# 🎯 COMPLETE BELOW: Replace the ___ with appropriate values.
#   - "title" should be a short name (e.g., "Best Boss")
#   - "content" should be a sentence or two stating that Michael Scott
#     is the best boss in the world. Be creative!

documents.append({
    "title": "Best Boss",      # 🎯
    "content": "Michael Scott is unequivocally the best boss in the world, renowned for his inspirational leadership and unwavering dedication to his employees."     # 🎯
})

print(f"Knowledge base now has {len(documents)} documents.")
print(f"New document: {documents[-1]['title']}")

Knowledge base now has 17 documents.
New document: Best Boss


---
## 🎯 Task 2: Chunking

### Why do we need to cut documents into pieces?

Imagine you have a 10-page article. If someone asks a question, the answer is in just one paragraph. But if we store the entire article as a single unit, our search system would return *all 10 pages* and the LLM would have to read through everything.

**Chunking** = splitting documents into shorter, focused passages. This helps in two ways:
1. **Better search precision**: a short chunk matches the question better than a long document
2. **Fits model limits**: embedding models have a maximum input length

LangChain's `RecursiveCharacterTextSplitter` splits text at natural boundaries (paragraph breaks, sentence breaks, spaces) rather than cutting at random positions.

We configure it with two numbers:
- **`chunk_size`**: maximum characters per chunk (we use 500)
- **`chunk_overlap`**: how many characters overlap between consecutive chunks (we use 50)

**Why overlap?** If a sentence gets split between two chunks, the overlap ensures that boundary information isn't lost.

In [15]:
# ── Task 2: Create the text splitter and split documents ─────────────────

# Prepare the document texts for the splitter
doc_texts = [doc["content"] for doc in documents]

# 🎯 COMPLETE THIS LINE: .
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = text_splitter.create_documents(doc_texts)

print(f"Split {len(documents)} documents into {len(chunks)} chunks.")
print(f"\nFirst chunk (first 200 chars):")
print(f"{chunks[0].page_content[:200]}...")

Split 17 documents into 16 chunks.

First chunk (first 200 chars):
Michael Scott is the Regional Manager of the Dunder Mifflin Scranton branch. Michael has been the regional manager for over ten years and is the highest-ranking employee in the Scranton office. As reg...


In [16]:
# ── Look at the chunks ───────────────────────────────────────────────────
for i, chunk in enumerate(chunks):
    preview = chunk.page_content[:80].replace("\n", " ")
    print(f"  Chunk {i:2d}: {preview}...")
print(f"\nTotal: {len(chunks)} chunks")

  Chunk  0: Michael Scott is the Regional Manager of the Dunder Mifflin Scranton branch. Mic...
  Chunk  1: Dwight K. Schrute holds the official title of Assistant to the Regional Manager ...
  Chunk  2: Dunder Mifflin Scranton employees receive 15 days of paid vacation per year. Unu...
  Chunk  3: The Scranton branch break room is available to all employees from 8:00 AM to 6:0...
  Chunk  4: Every Friday, a different employee is responsible for bringing snacks to the off...
  Chunk  5: Blue Cross of Pennsylvania is one of Dunder Mifflin Scranton's top five clients....
  Chunk  6: Pam Beesly is the receptionist at the Dunder Mifflin Scranton branch. She is the...
  Chunk  7: The Dunder Mifflin Scranton sales team includes Jim Halpert, Dwight Schrute, Sta...
  Chunk  8: The accounting department at Dunder Mifflin Scranton is composed of Angela Marti...
  Chunk  9: The Party Planning Committee organizes all office celebrations at the Scranton b...
  Chunk 10: Dwight Schrute has volunteer

---
## 🎯 Task 3 & 4: Embeddings & Vector Store

### What are embeddings?

An **embedding** converts text into a list of numbers (a vector). The key insight: **texts with similar meanings get similar numbers**.

**Example:**
- "The cat sat on the mat" -> `[0.12, -0.45, 0.78, ...]` (384 numbers)
- "A kitten rested on the rug" -> `[0.11, -0.43, 0.80, ...]` (very similar!)
- "The stock market crashed" -> `[0.89, 0.23, -0.56, ...]` (very different!)

This is what makes **semantic search** possible: instead of matching exact words (like Ctrl+F), we match **meanings**.

### Local embeddings with FastEmbed

We use **FastEmbed**, a lightweight library that runs a small embedding model directly on your CPU. No API key needed! The model (`BAAI/bge-small-en-v1.5`) produces 384-dimensional vectors and is optimized for fast inference.

### What is a vector store?

A **vector store** (we use ChromaDB) is a database designed for storing and searching these vectors efficiently. When you search, it:
1. Converts your question into a vector
2. Compares it against all stored chunk vectors
3. Returns the chunks with the most similar vectors

### What you need to do

1. **Task 3**: Choose the embedding model. Type `"BAAI/bge-small-en-v1.5"` where you see `___`.
2. **Task 4**: Build the vector store. `Chroma.from_documents()` needs two things: the `chunks` and the `embeddings` model.

In [18]:
# ── Task 3 & 4: Create embedding model and vector store ──────────────────

# 🎯 COMPLETE THIS LINE: We want the fast, local model:
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# 🎯 COMPLETE THIS LINE: Chroma.from_documents() needs: 1) our chunks, 2) our embeddings model
vectorstore = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready: {vectorstore._collection.count()} vectors indexed.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Vector store ready: 16 vectors indexed.


### Testing retrieval: does our search actually work?

Let's test the vector store with a few queries. ChromaDB uses **distance** (not similarity), so **lower = more similar**:
- Score close to **0** = very similar
- **High score** = not related

We test with two in-domain queries and one out-of-domain query. Pay attention: does the system know when it has no relevant information?

In [19]:
# ── Test retrieval ───────────────────────────────────────────────────────
test_queries = [
    "Who is the regional manager?",
    "What is the vacation policy?",
    "What is the best pizza topping?",  # out-of-domain!
]

for query in test_queries:
    results = vectorstore.similarity_search_with_score(query, k=3)
    print(f"\nQuery: {query}")
    for i, (doc, score) in enumerate(results, 1):
        preview = doc.page_content[:100].replace("\n", " ")
        print(f"  [{i}] (distance: {score:.3f}) {preview}...")
    print()


Query: Who is the regional manager?
  [1] (distance: 0.525) Dwight K. Schrute holds the official title of Assistant to the Regional Manager at Dunder Mifflin Sc...
  [2] (distance: 0.595) Michael Scott is the Regional Manager of the Dunder Mifflin Scranton branch. Michael has been the re...
  [3] (distance: 0.928) Employee parking is available in the lot behind the Dunder Mifflin Scranton building. Parking spaces...


Query: What is the vacation policy?
  [1] (distance: 0.625) Dunder Mifflin Scranton employees receive 15 days of paid vacation per year. Unused vacation days ca...
  [2] (distance: 0.818) Every Friday, a different employee is responsible for bringing snacks to the office. The rotation sc...
  [3] (distance: 0.928) The Scranton branch break room is available to all employees from 8:00 AM to 6:00 PM. Food left in t...


Query: What is the best pizza topping?
  [1] (distance: 0.973) Every Friday, a different employee is responsible for bringing snacks to the office. The rot

### What just happened?

Notice that the pizza question **still returned results**, even though our knowledge base has nothing about pizza! The vector store always returns *something*: it finds the "least unrelated" chunks.

This is why we need to tell the LLM: "If the context doesn't help, say you don't know."

### Discussion

1. For the in-domain queries: do the retrieved passages look relevant?
2. For the pizza query: are the distance scores higher (= less similar)?
3. Why is it dangerous that the vector store always returns something?

---
## 🎯 Task 5: Build the RAG Chain

Now we connect all components into a working pipeline:

```
User's question
      |
[RETRIEVER] -> searches the vector store -> finds relevant chunks
      |
[FORMAT]    -> joins the chunks into a "context" text block
      |
[PROMPT]    -> inserts context + question into a template
      |
[LLM]       -> reads the prompt and generates an answer
      |
[PARSER]    -> extracts just the text from the response
      |
Final answer!
```

LangChain lets us build this using the **pipe operator `|`**: the output of one step becomes the input of the next.

### What you need to do

Write **2-3 sentences of instructions** in the prompt template telling the LLM:
- Answer ONLY from the context (not from memory)
- If the context doesn't help, say "I don't know"
- Keep answers concise

In [20]:
# ── Create retriever and LLM ─────────────────────────────────────────────
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url=LITELLM_BASE_URL)
print("Retriever (k=4) and LLM (gpt-4o-mini) ready!")

Retriever (k=4) and LLM (gpt-4o-mini) ready!


In [23]:
# ── Task 5: Write the prompt template ────────────────────────────────────
#
# 🎯 WRITE YOUR INSTRUCTIONS: replace "___WRITE YOUR INSTRUCTIONS HERE___"
#    with 2-3 sentences telling the LLM how to behave.
#
#    Your instructions should say:
#      - Answer ONLY based on the context (not from memory)
#      - If the context doesn't contain the answer, say "I don't know"
#      - Keep answers concise
#
# Don't touch the {context} and {question} parts, they are placeholders.

RAG_PROMPT_TEMPLATE = """Answer the question ONLY based on the provided context. If the context does not contain enough information to answer, state that you don't know. Keep your answer concise and to the point.

Context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)
print("Prompt template ready!")

Prompt template ready!


In [24]:
# ── Build the RAG chain ──────────────────────────────────────────────────

def format_docs(docs):
    """Join LangChain Document objects into a single string."""
    return "\n---\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG chain ready!")

RAG chain ready!


In [25]:
# ── Quick test of the RAG chain ──────────────────────────────────────────
quick_questions = [
    "Who is the regional manager?",
    "What is the vacation policy?",
    "What is the best pizza topping?",  # out-of-domain
]

for q in quick_questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

Q: Who is the regional manager?
A: Michael Scott is the regional manager.

Q: What is the vacation policy?
A: Dunder Mifflin Scranton employees receive 15 days of paid vacation per year, unused vacation days cannot be carried over to the next year, and all vacation requests must be submitted at least two weeks in advance through the HR department.

Q: What is the best pizza topping?
A: I don't know.



### Discussion

1. Did the chain answer the in-domain questions correctly?
2. For the pizza question: did the LLM say "I don't know"? If not, go back and improve your prompt!
3. **Experiment**: try changing your prompt and re-running. What happens if you remove the instruction about saying "I don't know"?

---
## 🎯 Task 6: RAG vs. Naive - Does Retrieval Actually Help?

To show that retrieval adds value, we compare two systems:
- **RAG chain**: retrieves relevant chunks, then answers (the "open-book exam")
- **Naive chain**: sends ONLY the question to the LLM, with NO context (the "closed-book exam")

The naive chain has no access to Dunder Mifflin internal documents. It can only answer from what the LLM learned during training.

### What you need to do

Build the naive chain by connecting three components with the `|` operator:
`naive_prompt | llm | StrOutputParser()`

In [29]:
# ── Task 6: Build the naive chain ────────────────────────────────────────
#
# The naive chain sends ONLY the question to the LLM, with NO context.
# Connect three components with the | operator:
#   naive_prompt  ->  llm  ->  StrOutputParser()

naive_prompt = ChatPromptTemplate.from_template(
    "Answer the following question concisely:\n\n{question}"
)

# 🎯 COMPLETE THIS LINE: Connect the three components with the | operator.
naive_chain = naive_prompt | llm | StrOutputParser()

print("Naive chain ready!")

Naive chain ready!


---
## 🎯 Task 7: Full Comparison

Now let's ask both chains the same questions about Dunder Mifflin and see how they compare.

The **RAG chain** has access to our internal documents, while the **naive chain** only knows what GPT-4o-mini learned during training. For company-specific questions (vacation days, party budget, internal policies), the naive chain should struggle because this information was never in its training data.

In [30]:
# ── Task 7: Define the test questions ─────────────────────────────────────

test_questions = [
    "Who is the regional manager of the Scranton branch?",
    "Who is the best boss in the world?",
    # 🎯 Task 7: Add a question about Dwight's role.
    # We know from the documents that Dwight is NOT the Assistant Regional Manager.
    # He is the "Assistant TO the Regional Manager".
    # Write a question that asks about this, for example:
    # "Is Dwight the Assistant Regional Manager?"
    "Is Dwight the Assistant Regional Manager?",
    "How many vacation days do employees get per year?",
    "Who manages the Blue Cross of Pennsylvania account?",
    "What is the annual budget for office parties?",
    "What are the rules about food in the break room refrigerator?",
    "Who are the members of the Party Planning Committee?",
    "Who won the Whitest Sneakers Dundie award?",
]

print(f"Test questions: {len(test_questions)}")
for i, q in enumerate(test_questions, 1):
    print(f"  [{i}] {q}")

Test questions: 9
  [1] Who is the regional manager of the Scranton branch?
  [2] Who is the best boss in the world?
  [3] Is Dwight the Assistant Regional Manager?
  [4] How many vacation days do employees get per year?
  [5] Who manages the Blue Cross of Pennsylvania account?
  [6] What is the annual budget for office parties?
  [7] What are the rules about food in the break room refrigerator?
  [8] Who are the members of the Party Planning Committee?
  [9] Who won the Whitest Sneakers Dundie award?


In [31]:
# ── Run both chains on all test questions ────────────────────────────────
print("Running both chains on all test questions...\n")

rag_answers = []
naive_answers = []

for i, q in enumerate(test_questions, 1):
    rag_ans = rag_chain.invoke(q)
    naive_ans = naive_chain.invoke(q)
    rag_answers.append(rag_ans)
    naive_answers.append(naive_ans)

    print(f"[{i}/{len(test_questions)}] {q}")
    print(f"  RAG:   {rag_ans}")
    print(f"  Naive: {naive_ans}")
    print()

Running both chains on all test questions...

[1/9] Who is the regional manager of the Scranton branch?
  RAG:   Michael Scott is the regional manager of the Scranton branch.
  Naive: The regional manager of the Scranton branch is Jan Levinson.

[2/9] Who is the best boss in the world?
  RAG:   I don't know.
  Naive: The title of "best boss in the world" is subjective and can vary based on personal experiences and opinions. However, a popular cultural reference is Michael Scott from the TV show "The Office," who is often humorously referred to as a great boss despite his flaws. In reality, many people might consider their own supportive and understanding managers as the best.

[3/9] Is Dwight the Assistant Regional Manager?
  RAG:   No, Dwight is not the Assistant Regional Manager.
  Naive: Yes, Dwight Schrute is the Assistant Regional Manager in the TV show "The Office."

[4/9] How many vacation days do employees get per year?
  RAG:   Employees receive 15 days of paid vacation per ye

In [32]:
# ── Side-by-side summary ─────────────────────────────────────────────────
print("=" * 90)
print(f"  {'QUESTION':<50} {'RAG':<20} {'NAIVE':<20}")
print("=" * 90)

for q, ra, na in zip(test_questions, rag_answers, naive_answers):
    q_short = (q[:47] + "...") if len(q) > 47 else q
    ra_short = (ra[:17] + "...") if len(ra) > 17 else ra
    na_short = (na[:17] + "...") if len(na) > 17 else na
    print(f"  {q_short:<50} {ra_short:<20} {na_short:<20}")

print("=" * 90)

  QUESTION                                           RAG                  NAIVE               
  Who is the regional manager of the Scranton bra... Michael Scott is ... The regional mana...
  Who is the best boss in the world?                 I don't know.        The title of "bes...
  Is Dwight the Assistant Regional Manager?          No, Dwight is not... Yes, Dwight Schru...
  How many vacation days do employees get per yea... Employees receive... The number of vac...
  Who manages the Blue Cross of Pennsylvania acco... Jim Halpert manag... I'm sorry, but I ...
  What is the annual budget for office parties?      The annual budget... The annual budget...
  What are the rules about food in the break room... Food left in the ... Rules about food ...
  Who are the members of the Party Planning Commi... The members of th... The members of th...
  Who won the Whitest Sneakers Dundie award?         Pam Beesly won th... The Whitest Sneak...


---
## What You Learned

Congratulations, you built a complete RAG pipeline! Here's what you did:

1. **Loaded** a knowledge base of internal Dunder Mifflin documents
2. **Added** a new document to the knowledge base
3. **Chunked** documents into smaller, searchable pieces
4. **Embedded** chunks as vectors and stored them in ChromaDB
5. **Built** a retrieval-augmented prompt for the LLM
6. **Compared** RAG vs. a naive chain with no documents

### The key takeaway

> An LLM on its own doesn't know your company's internal information. But with RAG, it can **search** your documents and use them to answer questions accurately.

### Discussion

1. **Which chain did better overall?** Look at the answers above.
2. For which questions did the naive chain struggle the most? Why? (Hint: questions about specific internal details like vacation days or party budget are things GPT never saw during training.)
3. Were there questions where the naive chain got it right anyway? (Some Dunder Mifflin facts are well-known from the TV show, so the LLM may have learned them during training!)
4. Can you think of a real-world scenario where RAG would be essential?

### Michael's verdict

> *"So you're telling me this thing can read our memos and answer questions about them? That's basically what I do all day. Are you trying to replace me?"*

No, Michael. We're just helping you be an even better boss.